In [1]:
!pip install datasets

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

# ============ CONFIGURATION ============
# Training dataset path
TRAIN_DATA_PATH = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\all_groups\group_combined_features.csv"

# Test data path with rich features
TEST_DATA_PATH = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\1_testing_dataset_g21_mapped_processed_featured.csv"

# Model to use
MODEL_NAME = "roberta-large"
# ======================================

# Load datasets directly
print(f"Loading training data from: {TRAIN_DATA_PATH}")
train_df = pd.read_csv(TRAIN_DATA_PATH)
train_df['label'] = train_df['emotion_code']
print(f"Training set has {len(train_df)} samples")

print(f"Loading test data from: {TEST_DATA_PATH}")
test_df = pd.read_csv(TEST_DATA_PATH)
test_df['label'] = test_df['emotion_code']
print(f"Test set has {len(test_df)} samples")

# Load tokenizer
print(f"Loading {MODEL_NAME} tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Prepare datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

# Tokenize
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Load model
print(f"Loading {MODEL_NAME} model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=7)

# Training arguments
training_args = TrainingArguments(
    output_dir="./emotion_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
)

# Train
print("Starting fine-tuning...")
trainer.train()

# Evaluate
print("Evaluating on test set...")
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# Get predictions
predictions = trainer.predict(test_tokenized)
preds = np.argmax(predictions.predictions, axis=1)

# Emotion map for reporting
emotion_map = {
    0: 'happiness',
    1: 'sadness', 
    2: 'anger',
    3: 'fear',
    4: 'surprise',
    5: 'disgust',
    6: 'neutral'
}

# Print classification report
print("\nClassification Report:")
print(classification_report(
    test_df['label'], 
    preds, 
    target_names=[emotion_map[i] for i in range(7)]
))

# Confusion matrix
print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(test_df['label'], preds)
print(conf_matrix)

# Save model
model_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\models\roberta_emotion_classifier"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")

Loading training data from: C:\Users\ronle\Desktop\BUAS\Y2C\datasets\all_groups\group_combined_features.csv
Training set has 24295 samples
Loading test data from: C:\Users\ronle\Desktop\BUAS\Y2C\datasets\1_testing_dataset_g21_mapped_processed_featured.csv
Test set has 860 samples
Loading roberta-large tokenizer...


Map: 100%|██████████| 860/860 [00:00<00:00, 22053.42 examples/s]


Loading roberta-large model...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\ronle\anaconda3\envs\y2c\lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,1.404900,1.932687
2,1.397700,2.108709
3,1.403700,2.244946


Evaluating on test set...


Evaluation results: {'eval_loss': 1.9326870441436768, 'eval_runtime': 164.153, 'eval_samples_per_second': 5.239, 'eval_steps_per_second': 0.658, 'epoch': 3.0}

Classification Report:
              precision    recall  f1-score   support

   happiness       0.00      0.00      0.00       391
     sadness       0.00      0.00      0.00        65
       anger       0.00      0.00      0.00         1
        fear       0.00      0.00      0.00        53
    surprise       0.00      0.00      0.00        38
     disgust       0.00      0.00      0.00        36
     neutral       0.32      1.00      0.49       276

    accuracy                           0.32       860
   macro avg       0.05      0.14      0.07       860
weighted avg       0.10      0.32      0.16       860


Confusion Matrix:
[[  0   0   0   0   0   0 391]
 [  0   0   0   0   0   0  65]
 [  0   0   0   0   0   0   1]
 [  0   0   0   0   0   0  53]
 [  0   0   0   0   0   0  38]
 [  0   0   0   0   0   0  36]
 [  0   0   0  

c:\Users\ronle\anaconda3\envs\y2c\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ronle\anaconda3\envs\y2c\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ronle\anaconda3\envs\y2c\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model saved to C:\Users\ronle\Desktop\BUAS\Y2C\models\roberta_emotion_classifier
